# Lab 6: Motion and Image Recognition

In [1]:
import cv2
import numpy as np
import os

dataDir = '../images'

### 1. Optical Flow with Lucas-Kanade Algorithm

In [3]:
# Define a VideoCapture Object
cap = cv2.VideoCapture(0)
if not cap.isOpened():
    print("Cannot open camera")
    exit()

# Define maximum number of features to track
max_num_features = 100

# Create some random colors
color = np.random.randint(0, 255, (max_num_features, 3))

# Take first frame and find corners in it
ret, old_frame = cap.read()
old_gray = cv2.cvtColor(old_frame, cv2.COLOR_BGR2GRAY)
p0 = cv2.goodFeaturesToTrack(old_gray, mask=None, maxCorners=max_num_features, qualityLevel=0.3, minDistance=7, blockSize=7)

# Create a mask image for drawing purposes
mask = np.zeros_like(old_frame)
 
while True:
    ret, frame = cap.read()
    if not ret:
        print("Can't receive frame...")
        break
 
    frame_gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
 
    # Calculate optical flow
    p1, st, err = cv2.calcOpticalFlowPyrLK(old_gray, frame_gray, p0, None, winSize=(15, 15), maxLevel=2, criteria=(cv2.TERM_CRITERIA_EPS|cv2.TERM_CRITERIA_COUNT, 10, 0.03))
 
    # Select good points
    if p1 is not None:
        good_new = p1[st==1]
        good_old = p0[st==1]
 
    # Draw the tracks
    for i, (new, old) in enumerate(zip(good_new, good_old)):
        a, b = new.ravel()
        c, d = old.ravel()
        mask = cv2.line(mask, (int(a), int(b)), (int(c), int(d)), color[i].tolist(), 2)
        frame = cv2.circle(frame, (int(a), int(b)), 5, color[i].tolist(), -1)
    img = cv2.add(frame, mask)
 
    cv2.imshow('frame', img)
    if cv2.waitKey(1) != -1:
        break
 
    # Now update the previous frame and previous points
    old_gray = frame_gray.copy()
    p0 = good_new.reshape(-1, 1, 2)

cap.release()
cv2.destroyAllWindows()

**Exercise 1.1**: Replace ShiTomasi feature detection by FAST feature detector.

In [ ]:
# TODO

# 2. Image Recognition

In this notebook, we build an image classification pipeline using HOG (Histogram of Oriented Gradients) for feature extraction and SVM for classification. By the end of the lab, students should be able to extract HOG features from images, train and evaluate classifiers using OpenCV's `cv2.ml` module and `scikit-learn`, and compare different approaches in terms of accuracy.


In [ ]:
import numpy as np
import cv2
import matplotlib.pyplot as plt
from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, confusion_matrix, ConfusionMatrixDisplay, classification_report

### Loading and Exploring the Dataset

We will use the [MNIST](https://en.wikipedia.org/wiki/MNIST_database) handwritten digit dataset, which contains 70,000 grayscale images of digits (0–9), each of size 28×28 pixels.

In [ ]:
# Load MNIST dataset
mnist = fetch_openml('mnist_784', version=1, as_frame=False)

# Extract images and labels
images = mnist.data.astype(np.uint8)
labels = mnist.target.astype(np.int32)

print(f"Dataset size: {images.shape[0]} images")
print(f"Image vector length: {images.shape[1]} (i.e. {int(np.sqrt(images.shape[1]))}x{int(np.sqrt(images.shape[1]))} pixels)")
print(f"Labels: {np.unique(labels)}")

In [ ]:
# Reshape flat vectors back into 28x28 images
images_2d = images.reshape(-1, 28, 28)

# Visualize some samples from the dataset
f, axarr = plt.subplots(2, 5, figsize=(12, 5))
for i in range(10):
    # Find the first occurrence of each digit
    idx = np.where(labels == i)[0][0]
    ax = axarr[i // 5, i % 5]
    ax.imshow(images_2d[idx], cmap='gray', vmin=0, vmax=255)
    ax.set_title(f"Label: {labels[idx]}")
    ax.axis("off")
plt.suptitle("Sample Images from MNIST")
plt.tight_layout()

In [ ]:
# Use a subset to speed up training (10,000 samples)
subset_size = 10000
images_subset = images_2d[:subset_size]
labels_subset = labels[:subset_size]

# Split into training and testing sets (80% train, 20% test)
X_train, X_test, y_train, y_test = train_test_split(
    images_subset, labels_subset, test_size=0.2, random_state=42, stratify=labels_subset
)

print(f"Training set: {X_train.shape[0]} images")
print(f"Test set: {X_test.shape[0]} images")

### Feature Extraction with HOG

The [Histogram of Oriented Gradients (HOG)](https://en.wikipedia.org/wiki/Histogram_of_oriented_gradients) descriptor captures the distribution of gradient orientations in localized regions of an image. It is widely used for object detection and image classification.

Key parameters:
* **winSize**: the size of the detection window (must match the image size)
* **blockSize**: the size of the block used for normalization
* **blockStride**: the stride between adjacent blocks
* **cellSize**: the size of each cell in which gradient histograms are computed
* **nbins**: the number of orientation bins in the histogram

In [ ]:
# Define HOG descriptor parameters
winSize = (28, 28)
blockSize = (14, 14)
blockStride = (7, 7)
cellSize = (7, 7)
nbins = 9

# Create the HOG descriptor
hog = cv2.HOGDescriptor(winSize, blockSize, blockStride, cellSize, nbins)

# Compute HOG descriptor for a single image to inspect its shape
sample_descriptor = hog.compute(X_train[0])
print(f"HOG descriptor length: {sample_descriptor.shape[0]}")

In [ ]:
def compute_hog_features(images, hog_descriptor):
    """Compute HOG features for a set of images."""
    hog_features = []
    for img in images:
        descriptor = hog_descriptor.compute(img)
        hog_features.append(descriptor.flatten())
    return np.array(hog_features, dtype=np.float32)

# Compute HOG features for training and test sets
print("Computing HOG features for training set...")
X_train_hog = compute_hog_features(X_train, hog)

print("Computing HOG features for test set...")
X_test_hog = compute_hog_features(X_test, hog)

print(f"Training HOG features shape: {X_train_hog.shape}")
print(f"Test HOG features shape: {X_test_hog.shape}")

### Training an SVM Classifier

We will use OpenCV's [SVM](https://docs.opencv.org/4.x/d1/d2d/classcv_1_1ml_1_1SVM.html) implementation to train a classifier on the HOG features.

The Support Vector Machine (SVM) finds an optimal hyperplane that separates the data in feature space. Key parameters:
* **Kernel type**: defines how the data is mapped to a higher-dimensional space (e.g., LINEAR, RBF)
* **C**: regularization parameter that controls the trade-off between margin width and classification errors

In [ ]:
# Create and configure the SVM
svm = cv2.ml.SVM_create()
svm.setType(cv2.ml.SVM_C_SVC)
svm.setKernel(cv2.ml.SVM_LINEAR)
svm.setC(2.67)

# Train the SVM on HOG features
print("Training SVM...")
svm.train(X_train_hog, cv2.ml.ROW_SAMPLE, y_train)
print("Training complete!")

### Evaluation

In [ ]:
# Predict on the test set
_, y_pred = svm.predict(X_test_hog)
y_pred = y_pred.flatten().astype(np.int32)

# Calculate accuracy
accuracy = accuracy_score(y_test, y_pred)
print(f"Test Accuracy: {accuracy * 100:.2f}%")

In [ ]:
# Display the classification report
print(classification_report(y_test, y_pred))

In [ ]:
# Compute and display the confusion matrix
cm = confusion_matrix(y_test, y_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=np.arange(10))

f, ax = plt.subplots(figsize=(8, 8))
disp.plot(ax=ax, cmap='Blues', colorbar=False)
_ = ax.set_title("Confusion Matrix")

In [ ]:
# Visualize some correct and incorrect predictions
correct = np.where(y_pred == y_test)[0]
incorrect = np.where(y_pred != y_test)[0]

f, axarr = plt.subplots(2, 5, figsize=(14, 6))

# Show 5 correct predictions
for i in range(5):
    idx = correct[i]
    axarr[0, i].imshow(X_test[idx], cmap='gray', vmin=0, vmax=255)
    axarr[0, i].set_title(f"Pred: {y_pred[idx]}")
    axarr[0, i].axis("off")
axarr[0, 0].set_ylabel("Correct", fontsize=12)

# Show 5 incorrect predictions
for i in range(5):
    idx = incorrect[i]
    axarr[1, i].imshow(X_test[idx], cmap='gray', vmin=0, vmax=255)
    axarr[1, i].set_title(f"Pred: {y_pred[idx]} (GT: {y_test[idx]})")
    axarr[1, i].axis("off")
axarr[1, 0].set_ylabel("Incorrect", fontsize=12)

plt.suptitle("Sample Predictions")
plt.tight_layout()

### Exercises

**Exercise 2.1**: Train the SVM using raw pixel values (flattened 28×28 images) as features instead of HOG descriptors. Compare the accuracy with the HOG-based approach.

**Tip**: Normalize the pixel values to the range [0, 1] by dividing by 255 before training.

In [ ]:
# TODO

**Exercise 2.2**: Investigate the impact of the training set size on the model's performance. Train the SVM with 1,000, 5,000, 10,000, and 20,000 samples and plot the test accuracy as a function of the number of training samples.

**Tip**: Use `matplotlib` to create a line plot of the results.

In [ ]:
# TODO

**Exercise 2.3**: Apply image preprocessing before extracting HOG features. Try the following and compare results:
* Apply Gaussian blur with a 3×3 kernel
* Resize the images to 32×32 pixels (remember to adjust `winSize` in the HOG descriptor accordingly)
* Apply histogram equalization using `cv2.equalizeHist()`

In [ ]:
# TODO

**Exercise 2.4**: Implement a simple test by drawing a digit on a blank 28×28 image and classifying it with the trained model.

**Tips**:
* Create a blank image with `np.zeros((28, 28), dtype=np.uint8)`
* Use `cv2.line()`, `cv2.circle()`, or `cv2.putText()` to draw a digit
* Compute the HOG descriptor and use `svm.predict()` to classify it

In [ ]:
# TODO

### Additional challenge

Replace the SVM classifier with a different OpenCV ML model and compare the results. Try one or both of the following:
* [KNN](https://docs.opencv.org/4.x/dd/de1/classcv_1_1ml_1_1KNearest.html) (`cv2.ml.KNearest_create()`): set the default number of neighbours with `setDefaultK()` and train with `train()`. Experiment with different values of K (e.g., 3, 5, 7).
* [MLP (Multilayer Perceptron)](https://docs.opencv.org/4.x/d0/dce/classcv_1_1ml_1_1ANN__MLP.html) (`cv2.ml.ANN_MLP_create()`): define the network architecture with `setLayerSizes()` (e.g., input layer matching the HOG descriptor length, one hidden layer with 128 neurons, and an output layer with 10 neurons). Set the activation function with `setActivationFunction()` and the training method with `setTrainMethod()`.

Compare the test accuracy with the SVM baseline.

**Tips**:
* Both classifiers use the same `train()` and `predict()` interface as `cv2.ml.SVM`
* For the MLP, labels must be provided as a one-hot encoded matrix (e.g., label 3 → `[0, 0, 0, 1, 0, 0, 0, 0, 0, 0]`). Use `np.eye(10)[labels]` to convert.
* For the MLP, `predict()` returns continuous values for each class — use `np.argmax()` on each row to obtain the predicted label

In [ ]:
# TODO